In [1]:
import fitz

In [2]:
import fitz
import json
import asyncio
import time
import sys
import re
import binascii
import unicodedata
import ollama
import nest_asyncio
import logging
from typing import List, Set, Optional, Dict
from hashlib import blake2b
from collections import Counter

# 1. INTENSIVE LOGGING SETUP
# Set to DEBUG to see the most granular information
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("DeepPipeline")

nest_asyncio.apply()

class TextGuardrails:
    def __init__(self, semantic_threshold: float = 0.85, num_perm: int = 64):
        self.threshold = semantic_threshold
        self.num_perm = num_perm
        self.exact_hashes: Set[str] = set()
        self.minhash_registry: List[List[int]] = []

    def is_duplicate(self, text: str) -> bool:
        h = blake2b(text.encode(), digest_size=16).hexdigest()
        if h in self.exact_hashes:
            return True
        
        if len(text) < 20: return False 
        
        # MinHash check
        clean = "".join(text.lower().split())
        shingles = {clean[i:i + 5] for i in range(len(clean) - 4)}
        if not shingles: return False
        
        current_sig = [min(binascii.crc32(f"{s}{j}".encode()) & 0xffffffff for s in shingles) for j in range(self.num_perm)]

        for existing_sig in self.minhash_registry:
            match_count = sum(a == b for a, b in zip(current_sig, existing_sig))
            if (match_count / self.num_perm) >= self.threshold:
                return True

        self.exact_hashes.add(h)
        self.minhash_registry.append(current_sig)
        return False

    def apply(self, text: str) -> Optional[str]:
        text = unicodedata.normalize("NFKC", text)
        text = re.sub(r"[a-zA-Z0-9_.+-]+ email", "[EMAIL]", text) # Simplified PII for speed
        text = " ".join(text.split())
        
        sentences = re.split(r'(?<=[.!?]) +', text)
        unique_sentences = [s.strip() for s in sentences if s.strip() and not self.is_duplicate(s.strip())]
        
        result = " ".join(unique_sentences)
        return result if len(result) > 5 else None

class BatchPDFKeywordPipeline:
    def __init__(self, model_name="qwen2.5:7b", batch_size=6):
        self.model_name = model_name
        self.batch_size = batch_size
        self.semaphore = asyncio.Semaphore(2)

    async def process_batch(self, batch: List[Dict], batch_num: int):
        async with self.semaphore:
            logger.info(f"==> Batch {batch_num}: Sending {len(batch)} items to Ollama ({self.model_name})")
            try:
                prompt = (
                    "You are a Retail Banking Domain Expert. Return valid JSON only.\n"
                    "TASK: Assign EXACTLY 5 banking keywords per topic_id.\n"
                    f"DATA: {json.dumps(batch)}"
                )
                
                start_llm = time.time()
                response = await asyncio.wait_for(
                    asyncio.to_thread(
                        ollama.chat,
                        model=self.model_name,
                        format="json",
                        messages=[{"role": "user", "content": prompt}]
                    ), timeout=90.0
                )
                duration = time.time() - start_llm
                logger.info(f"==> Batch {batch_num}: Received LLM response in {duration:.2f}s")
                
                return json.loads(response["message"]["content"])
            except Exception as e:
                logger.error(f"!!! Batch {batch_num} FAILED: {str(e)}")
                return {}

    def extract_hierarchy(self, pdf_path: str) -> List[Dict]:
        logger.info(f"Step 1: Opening PDF at {pdf_path}")
        doc = fitz.open(pdf_path)
        spans = []
        
        logger.info(f"Step 1: Iterating through {len(doc)} pages...")
        for page_num, page in enumerate(doc, 1):
            blocks = page.get_text("dict")["blocks"]
            for b in blocks:
                if "lines" in b:
                    for l in b["lines"]:
                        for s in l["spans"]:
                            if s["text"].strip():
                                spans.append({"text": s["text"].strip(), "size": round(s["size"], 1)})
        
        logger.info(f"Step 1: Found {len(spans)} text spans. Analyzing font distribution...")
        size_counts = Counter([s["size"] for s in spans])
        unique_sizes = sorted(size_counts.keys(), reverse=True)
        
        logger.info(f"Step 1: Detected unique font sizes: {unique_sizes}")
        section_f = unique_sizes[0]
        topic_f = unique_sizes[1] if len(unique_sizes) > 1 else unique_sizes[0]
        logger.info(f"Step 1: Hierarchy Rule -> Section Size: {section_f}, Topic Size: {topic_f}")

        results, cur_sec_name, cur_sec_id = [], "Header", "S001"
        s_idx, t_idx = 1, 0

        for span in spans:
            if span["size"] == section_f and len(span["text"]) < 80:
                s_idx += 1
                cur_sec_name, cur_sec_id = span["text"], f"S{s_idx:03}"
                t_idx = 0
            elif span["size"] == topic_f and len(span["text"]) < 80:
                t_idx += 1
                results.append({
                    "section_id": cur_sec_id,
                    "section_name": cur_sec_name,
                    "topic_id": f"{cur_sec_id}_T{t_idx:03}",
                    "topic_name": span["text"],
                    "raw_lines": []
                })
            elif results:
                results[-1]["raw_lines"].append(span["text"])
        
        doc.close()
        logger.info(f"Step 1: Extraction complete. Identified {len(results)} potential topics.")
        return results

async def run_master_pipeline(pdf_path: str):
    overall_start = time.time()
    logger.info("PIPELINE INITIALIZED")
    
    pipeline = BatchPDFKeywordPipeline(batch_size=5)
    guardrails = TextGuardrails()

    # 1. Extraction
    raw_data = pipeline.extract_hierarchy(pdf_path)
    if not raw_data:
        logger.critical("No data found. Check PDF formatting or path.")
        return

    # 2. Cleaning
    logger.info("Step 2: Starting Text Cleaning and Deduplication...")
    processed_items = []
    for i, item in enumerate(raw_data):
        raw_string = " ".join(item.pop("raw_lines"))
        clean_text = guardrails.apply(raw_string)
        
        if clean_text:
            item["text"] = clean_text
            processed_items.append(item)
        
        if (i+1) % 20 == 0:
            logger.info(f"Step 2: Cleaned {i+1}/{len(raw_data)} items...")

    logger.info(f"Step 2: Finished. {len(processed_items)} items survived cleaning.")

    # 3. Keyword Generation
    total_items = len(processed_items)
    final_output = []
    logger.info(f"Step 3: Starting LLM Batch Processing (Batch Size: {pipeline.batch_size})")

    for i in range(0, total_items, pipeline.batch_size):
        batch_idx = (i // pipeline.batch_size) + 1
        batch = processed_items[i : i + pipeline.batch_size]
        
        # Progress Tracking
        progress = (i / total_items) * 100
        logger.info(f"Step 3: Progress: {progress:.1f}% | Handling items {i} to {min(i+pipeline.batch_size, total_items)}")
        
        llm_input = [{"topic_id": x["topic_id"], "text": x["text"][:600]} for x in batch]
        keywords_map = await pipeline.process_batch(llm_input, batch_idx)

        for item in batch:
            tid = item["topic_id"]
            item["keywords"] = keywords_map.get(tid, ["Banking", "Standard", "Retail"])
            if tid not in keywords_map:
                logger.warning(f"  -> Topic {tid} did not receive LLM keywords. Used defaults.")
            final_output.append(item)

    # 4. Save
    output_path = "structured_loan_data.json"
    logger.info(f"Step 4: Writing results to {output_path}")
    with open(output_path, "w") as f:
        json.dump(final_output, f, indent=4)
    
    total_duration = time.time() - overall_start
    logger.info(f"SUCCESS: Pipeline completed in {total_duration:.2f} seconds.")
    logger.info(f"Final Record Count: {len(final_output)}")

# RUN
PDF_FILE = r"F:\Loan_Details_RAG_System1\Data\loan_products_final.pdf"
await run_master_pipeline(PDF_FILE)

2026-04-13 11:53:53,826 | INFO | PIPELINE INITIALIZED
2026-04-13 11:53:53,826 | INFO | Step 1: Opening PDF at F:\Loan_Details_RAG_System1\Data\loan_products_final.pdf
2026-04-13 11:53:53,876 | INFO | Step 1: Iterating through 35 pages...
2026-04-13 11:53:54,049 | INFO | Step 1: Found 1832 text spans. Analyzing font distribution...
2026-04-13 11:53:54,052 | INFO | Step 1: Detected unique font sizes: [18.0, 13.4, 13.0, 12.5, 12.0, 10.1]
2026-04-13 11:53:54,057 | INFO | Step 1: Hierarchy Rule -> Section Size: 18.0, Topic Size: 13.4
2026-04-13 11:53:54,068 | INFO | Step 1: Extraction complete. Identified 138 potential topics.
2026-04-13 11:53:54,075 | INFO | Step 2: Starting Text Cleaning and Deduplication...
2026-04-13 11:53:55,137 | INFO | Step 2: Cleaned 20/138 items...
2026-04-13 11:53:56,150 | INFO | Step 2: Cleaned 40/138 items...
2026-04-13 11:53:57,036 | INFO | Step 2: Cleaned 60/138 items...
2026-04-13 11:53:58,420 | INFO | Step 2: Cleaned 80/138 items...
2026-04-13 11:53:59,080 |

In [3]:
with open("structured_loan_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [12]:
text = unicodedata.normalize("NFKC", data[4]["text"])

text

'The margin depends on the residual maturity period of the security: \uf0b7 NSC / KVP / Government Bonds o 15% of face value if residual maturity is less than 3 years o 20% of face value if residual maturity is 3 years or more \uf0b7 Life Insurance Policies o 15% of surrender value if maturity is within 3 years o 20% of surrender value if maturity is 3 years or more For staff borrowers against NSC, a concessional margin of 10% of face value applies.'